### Import libraries

In [1]:
import numpy as np
import pandas as pd

from utils.generate_synthetic_graph import SyntheticGraphGenerator
from utils.nalingam_environment import GraphEnvNALiNGAM

from utils.graph_functions import get_initial_subgraph

from cdt.metrics import precision_recall, SID, SHD

/home/malbo/NA-LiNGAM/.env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Detecting 2 CUDA device(s).


In [2]:
from utils.cdmodels.pc import PCAlgorithm
from utils.cdmodels.fci import FCIAlgorithm
from utils.cdmodels.ges import GESAlgorithm
from utils.cdmodels.boss import BOSSAlgorithm
from utils.cdmodels.gin import GINAlgorithm
from utils.cdmodels.exact_search import ExactSearchAlgorithm
from utils.cdmodels.grasp import GRaSPAlgorithm
from utils.cdmodels.granger import GrangerAlgorithm
from utils.nalingam_score import NALiNGAMScoreAlgorithm
from utils.cdmodels.notears import NOTEARSAlgorithm
from utils.cdmodels.cam import CAMAlgorithm
from utils.cdmodels.ccdr import CCDrAlgorithm
from utils.cdmodels.gies import GIESAlgorithm

### Prepare environment

In [3]:
starting_nodes = 3

dataset = SyntheticGraphGenerator(
    sample_size = 1000,
    node_size = 10,
    min_edges = 2,
    max_edges = 5,
    n_noise_nodes=5,
    seed=42,
    non_linear=False
)

dataset.generate_samples()

df = dataset.get_dataframe()
nodes = dataset.get_nodes()
real_graph = dataset.get_graph()

initial_variables = nodes[:starting_nodes]
initial_graph = get_initial_subgraph(real_graph, initial_variables)

In [4]:
method_list = [
   'NALiNGAMAlgorithm',
   'PCAlgorithm',
   'FCIAlgorithm',
   'GESAlgorithm',
   #'BOSSAlgorithm',
   'ExactSearchAlgorithm',
   'LiNGAMAlgorithm',
   'GRaSPAlgorithm',
   #'GrangerAlgorithm',
   'NOTEARSAlgorithm',
   'CAMAlgorithm',
   #'CCDrAlgorithm',
   'GIESAlgorithm'
]

In [5]:
# METHOD: [AUC, SHD, SID]
metric_results = {}

In [6]:
for discovery_method in method_list:
    print(f"Running discovery method: {discovery_method}")
    if discovery_method == 'NALiNGAMAlgorithm':
        env = GraphEnvNALiNGAM(df, initial_graph)
        found_state, _ = env.get_best_state_fast(30)

        graph = env.get_graph(found_state, iterations=10)
    elif discovery_method == 'LiNGAMAlgorithm':
        env = GraphEnvNALiNGAM(df, initial_graph)

        graph = env.get_graph(np.ones(len(df.columns) - len(list(initial_graph.nodes()))), iterations=1) # Using NA-LiNGAM with 1 iteration to simulate LiNGAM without score
    else:
        try:
            discovery_method_function = globals()[discovery_method]
            graph = discovery_method_function(df, df.columns).get_graph()
        except KeyError:
            print(f"Discovery method {discovery_method} not found.")
            continue
    
    auc = precision_recall(real_graph, graph)[0]
    shd = SHD(real_graph, graph)
    sid = SID(real_graph, graph)

    metric_results[discovery_method] = [auc, shd, sid]

Running discovery method: NALiNGAMAlgorithm
Running discovery method: PCAlgorithm


Depth=2, working on node 9: 100%|██████████| 10/10 [00:00<00:00, 81.05it/s] 


Running discovery method: FCIAlgorithm


Depth=0, working on node 9: 100%|██████████| 10/10 [00:00<00:00, 127.89it/s]


X3 --> X1
X1 --> X4
X5 --> X3
Running discovery method: GESAlgorithm
Running discovery method: ExactSearchAlgorithm
Running discovery method: LiNGAMAlgorithm
Running discovery method: GRaSPAlgorithm

GRaSP completed in: 0.36s 
Running discovery method: NOTEARSAlgorithm
Running discovery method: CAMAlgorithm
Running discovery method: GIESAlgorithm


In [7]:
for method in metric_results:
    print(f"{method}: AUC={metric_results[method][0]:.4f}, SHD={metric_results[method][1]}, SID={metric_results[method][2]}")

NALiNGAMAlgorithm: AUC=0.9289, SHD=2, SID=4.0
PCAlgorithm: AUC=0.4656, SHD=9, SID=15.0
FCIAlgorithm: AUC=0.5011, SHD=9, SID=18.0
GESAlgorithm: AUC=0.6800, SHD=9.0, SID=16.0
ExactSearchAlgorithm: AUC=0.1800, SHD=15, SID=19.0
LiNGAMAlgorithm: AUC=0.9289, SHD=2, SID=4.0
GRaSPAlgorithm: AUC=0.6800, SHD=9.0, SID=16.0
NOTEARSAlgorithm: AUC=0.6079, SHD=8, SID=16.0
CAMAlgorithm: AUC=0.4367, SHD=13, SID=16.0
GIESAlgorithm: AUC=0.6076, SHD=11, SID=20.0


In [8]:
# env = GraphEnvNALiNGAM(df, initial_graph)

# found_state, _ = env.get_best_state_fast(30)
# print("Found state:", found_state)

# graph = env.get_graph(found_state, iterations=10)

# auc = precision_recall(real_graph, graph)[0]
# sid = SID(real_graph, graph)
# shd = SHD(real_graph, graph)

# print(f"AUC: {auc:.4f}, SID: {sid}, SHD: {shd}")